# Getting additional context features

This notebook computes **context-level features** (perplexity, length, gzip compression) for overflow detection.

**Steps:**
1. Load base features from `features.jsonl`.
2. Load context text from a second file.
3. Compute perplexity, token/char length, and gzip bits-per-char for each context.
4. Save merged features to output JSONL.

In [8]:
# Config: set paths and CUDA device before running
BASE_PATH = "/app/xlong/scripts/data_preprocessing/runs/squad"   # root dir with mistral_trivia, mistral_squad, mistral_hotpot
DATASET = "squad"                        # dataset: trivia, squad, hotpot
CUDA_DEVICE = "0"                         # GPU device ID
XRAG_DIR = "../xRAG"               # path to xRAG source (src/model.py, etc.)

FEATURES_PATH = f"{BASE_PATH}/probe/features.jsonl"
SAMPLES_PATH = f"{BASE_PATH}/samples.jsonl"
OUTPUT_PATH = f"{BASE_PATH}/features_context_metrics.jsonl"

MODEL_PATH = "/app/models/xrag-7b"
MAX_LENGTH = 4096  # max context length for perplexity


In [2]:
import os, torch, numpy as np, sys
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE
sys.path.insert(0, XRAG_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
import json
import pandas as pd

# --- Load features ---
rows = []
with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_features = pd.DataFrame(rows)

# --- Load samples ---
rows = []
with open(SAMPLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_samples = pd.DataFrame(rows)

# Extract context from background list
df_samples["context_text"] = df_samples["background"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

# Keep only needed columns
df_samples = df_samples[["id", "context_text"]]

# --- Merge ---
df_features = df_features.merge(df_samples, on="id", how="left")

In [11]:
# Utility: merge without duplicating columns

def merge_on_id_no_dupes(df_left: pd.DataFrame, df_right: pd.DataFrame, key="id", prefer="right") -> pd.DataFrame:
    """
    Merge on `key` and avoid duplicated columns by keeping only non-overlapping columns from the other DF.
    prefer:
      - "right": keep df_right values for overlapping columns (drop overlaps from left before merge)
      - "left":  keep df_left values for overlapping columns (drop overlaps from right before merge)
    """
    assert key in df_left.columns and key in df_right.columns, f"'{key}' must exist in both DFs"

    left = df_left.copy()
    right = df_right.copy()

    overlap = (set(left.columns) & set(right.columns)) - {key}

    if prefer == "right":
        left = left.drop(columns=list(overlap), errors="ignore")
    elif prefer == "left":
        right = right.drop(columns=list(overlap), errors="ignore")
    else:
        raise ValueError("prefer must be 'right' or 'left'")

    merged = left.merge(right, on=key, how="left", validate="one_to_one")
    return merged


In [5]:
from transformers import AutoTokenizer
from src.model import SFR, XMistralForCausalLM
from src.language_modeling.utils import XRAG_TOKEN

model = XMistralForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",
    attn_implementation="eager",
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    padding_side="left",
    add_eos_token=False,
    use_fast=False,
)
if tokenizer.pad_token:
    pass
elif tokenizer.unk_token:
    tokenizer.pad_token_id = tokenizer.unk_token_id
elif tokenizer.eos_token:
    tokenizer.pad_token_id = tokenizer.eos_token_id


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [12]:
import math, zlib
from tqdm.auto import tqdm

def gzip_bits_per_char(text: str) -> float:
    """Info-density proxy using zlib compression."""
    if not text:
        return 0.0
    raw = text.encode("utf-8")
    comp = zlib.compress(raw)
    bits = len(comp) * 8.0
    return bits / max(1, len(text))

def bg_to_text(bg):
    if isinstance(bg, list):
        return "\n\n".join(map(str, bg))
    return str(bg)

@torch.no_grad()
def ppl_onepass(text: str, model, tokenizer, device="cuda", max_length=2048):
    text = (text or "").strip()
    if not text:
        return float("nan"), 0

    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    enc = {k: v.to(device) for k, v in enc.items()}

    out = model(input_ids=enc["input_ids"], attention_mask=enc.get("attention_mask"), labels=enc["input_ids"])

    loss = float(out.loss.detach().float().item())
    return float(math.exp(loss)), len(enc["input_ids"][0])


In [13]:
# Compute context features
model.eval()

df_merged = df_features.copy()
df_merged["context_text"] = df_merged["context_text"].apply(bg_to_text)

# Compute perplexity once per unique context
unique_ctx = df_merged["context_text"].dropna().unique().tolist()

ctx2ppl = {}
ctx2len = {}
for ctx in tqdm(unique_ctx, desc="Computing perplexity"):
    ctx2ppl[ctx], ctx2len[ctx] = ppl_onepass(ctx, model, tokenizer, device=str(device), max_length=MAX_LENGTH)

# Map back to all rows
df_merged["context_perplexity"] = df_merged["context_text"].map(ctx2ppl)
df_merged["context_length_tokens"] = df_merged["context_text"].map(ctx2len)
df_merged["context_gzip_bits_per_char"] = df_merged["context_text"].map(gzip_bits_per_char)
df_merged["context_length_chars"] = df_merged["context_text"].map(len)


Computing perplexity:   0%|          | 0/41 [00:00<?, ?it/s]

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


In [14]:
# Save merged features with context metrics
df_merged.to_json(OUTPUT_PATH, orient="records", lines=True)
print(f"Saved {len(df_merged)} rows to {OUTPUT_PATH}")

Saved 100 rows to /app/xlong/scripts/data_preprocessing/runs/squad/features_context_metrics.jsonl
